# Step 6: RQ2, can we predict tomorrow's unhealthy day?

Target: `y_unhealthy_next` = 1 if tomorrow's AQI > 200 (Poor, Very Poor or Severe).
Features: today's pollutants and AQI, yesterday's PM2.5/PM10/NO2/AQI, 7-day PM2.5 mean, calendar,
city (39 features, built in Step 4).

Protocol: tune each model on validation (2019) by ROC-AUC; choose the decision threshold on
validation; evaluate once on test (2020), also with the lockdown months removed.

**Threshold rule.** A missed unhealthy day (no warning, people exposed) costs more than a false
alarm, so the threshold maximises F2 on validation. F2 weights recall twice as much as precision.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
try:                                   # Colab: get the repo (code + data) first
    import google.colab  # noqa: F401
    if not (ROOT / "src" / "aq.py").exists():
        import subprocess
        subprocess.run(["git", "clone", "-b", "claude/wizardly-meitner-nm2sfq",
                        "https://github.com/s-aura-bhhh/Air-quality-analysis.git"], check=True)
        ROOT = Path.cwd() / "Air-quality-analysis"
except ImportError:
    pass
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import aq

FIG, TAB = ROOT / "report" / "figures", ROOT / "report" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

def save(name):
    plt.tight_layout(); plt.savefig(FIG / f"06_{name}.png", dpi=150); plt.show()

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
                             confusion_matrix, precision_score, recall_score, f1_score, fbeta_score,
                             accuracy_score)

In [2]:
data, info = aq.build_dataset()
F = info["rq2_features"]
c = data.dropna(subset=["y_unhealthy_next"]).sort_values("Date").reset_index(drop=True)
S = {p: c[c["period"] == p] for p in ["train", "val", "test"]}
S["test_nolock"] = S["test"][~S["test"]["lockdown"]]
X = {k: v[F] for k, v in S.items()}
y = {k: v["y_unhealthy_next"].astype(int).values for k, v in S.items()}
pd.DataFrame({k: {"rows": len(v), "positive_rate": round(y[k].mean(), 3)} for k, v in S.items()}).T

,rows,positive_rate
train,13141.0,0.336
val,5921.0,0.245
test,2963.0,0.122
test_nolock,1473.0,0.210


## 6.1 Baselines

In [3]:
def scores(y_true, y_pred, y_score=None):
    out = {"accuracy": accuracy_score(y_true, y_pred), "precision": precision_score(y_true, y_pred, zero_division=0),
           "recall": recall_score(y_true, y_pred), "F1": f1_score(y_true, y_pred),
           "F2": fbeta_score(y_true, y_pred, beta=2)}
    if y_score is not None:
        out["ROC_AUC"] = roc_auc_score(y_true, y_score)
        out["PR_AUC"] = average_precision_score(y_true, y_score)
    return out

def best_threshold(y_true, s):
    # threshold that maximises F2 (recall weighted twice) on the given set
    p, r, t = precision_recall_curve(y_true, s)
    f2 = 5 * p * r / np.maximum(4 * p + r, 1e-12)
    return t[int(np.nanargmax(f2[:-1]))]

# Persistence: tomorrow's label = today's label. If today's AQI is missing, predict 'not unhealthy'.
# As a score for ROC/PR, use today's AQI (higher AQI today -> more likely unhealthy tomorrow).
persist_pred = {k: v["unhealthy_today"].fillna(0).astype(int).values for k, v in S.items()}
persist_score = {k: v["aqi_z_filled"].values for k, v in S.items()}
print("rows where today's AQI is missing (persistence falls back to 0):",
      {k: int(v["unhealthy_today"].isna().sum()) for k, v in S.items()})
base = []
for part in ["val", "test", "test_nolock"]:
    base.append({"model": "Majority (always 0)", "set": part,
                 **scores(y[part], np.zeros_like(y[part]), np.zeros(len(y[part])))})
    base.append({"model": "Persistence", "set": part,
                 **scores(y[part], persist_pred[part], persist_score[part])})
# Fair version: today's AQI as the score, with its threshold tuned on validation like every model.
thr_today = best_threshold(y["val"], persist_score["val"])
today_pred = {k: (v >= thr_today).astype(int) for k, v in persist_score.items()}
for part in ["val", "test", "test_nolock"]:
    base.append({"model": "Today's AQI, tuned threshold", "set": part,
                 **scores(y[part], today_pred[part], persist_score[part])})
print(f"tuned threshold on today's AQI: {np.expm1(thr_today * info['aqi_log_std'] + info['aqi_log_mean']):.0f}")
base = pd.DataFrame(base).set_index(["model", "set"]).round(3)
base

rows where today's AQI is missing (persistence falls back to 0): {'train': 323, 'val': 84, 'test': 39, 'test_nolock': 20}
tuned threshold on today's AQI: 160


accuracy  precision  recall     F1     F2  ROC_AUC  PR_AUC
model                        set                                                                    
Majority (always 0)          val             0.755      0.000   0.000  0.000  0.000    0.500   0.245
Persistence                  val             0.927      0.852   0.848  0.850  0.849    0.976   0.935
Majority (always 0)          test            0.878      0.000   0.000  0.000  0.000    0.500   0.122
Persistence                  test            0.936      0.736   0.740  0.738  0.740    0.963   0.811
Majority (always 0)          test_nolock     0.790      0.000   0.000  0.000  0.000    0.500   0.210
Persistence                  test_nolock     0.910      0.787   0.787  0.787  0.787    0.958   0.854
Today's AQI, tuned threshold val             0.899      0.723   0.956  0.823  0.898    0.976   0.935
                             test            0.904      0.567   0.906  0.698  0.809    0.963   0.811
                             test_nolock     0.885      0.663   0.926  0.773  0.858    0.958   0.854

The majority classifier reaches high accuracy on test (most 2020 days are not unhealthy) with zero
recall: accuracy is useless here. Persistence is the bar to beat, and the fair version of it
is "today's AQI with a tuned threshold": every model below gets a threshold tuned on validation,
so the baseline must get one too. Otherwise a model can "beat persistence" on recall simply by
alarming at a lower AQI.

## 6.2 Models, tuned on validation by ROC-AUC

In [4]:
def tune(name, make, grid):
    rows = []
    for params in grid:
        m = make(**params).fit(X["train"], y["train"])
        rows.append({**params, "val_ROC_AUC": roc_auc_score(y["val"], m.predict_proba(X["val"])[:, 1]), "_m": m})
    t = pd.DataFrame(rows).sort_values("val_ROC_AUC", ascending=False)
    t.drop(columns="_m").round(4).to_csv(TAB / f"06_tuning_{name}.csv", index=False)
    print(f"{name:8s} best {t.iloc[0].drop(['_m', 'val_ROC_AUC']).to_dict()}  val AUC {t.iloc[0]['val_ROC_AUC']:.4f}")
    return t.iloc[0]["_m"]

scaled = lambda m: make_pipeline(StandardScaler(), m)
models = {
    "LogReg": tune("LogReg", lambda C: scaled(LogisticRegression(C=C, class_weight="balanced", max_iter=3000)),
                   [{"C": C} for C in [0.001, 0.01, 0.1, 1, 10, 100]]),
    "LDA": tune("LDA", lambda shrinkage: scaled(LinearDiscriminantAnalysis(solver="lsqr", shrinkage=shrinkage)),
                [{"shrinkage": s} for s in [None, "auto"]]),
    "Tree": tune("Tree", lambda max_depth: DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=20,
                                                                 class_weight="balanced", random_state=aq.SEED),
                 [{"max_depth": d} for d in range(2, 11)]),
    "RF": tune("RF", lambda max_depth, min_samples_leaf: RandomForestClassifier(
                   n_estimators=300, max_depth=max_depth, min_samples_leaf=min_samples_leaf,
                   class_weight="balanced", n_jobs=-1, random_state=aq.SEED),
               [{"max_depth": d, "min_samples_leaf": l} for d in [6, 10, None] for l in [1, 5]]),
    "KNN": tune("KNN", lambda k: scaled(KNeighborsClassifier(k, n_jobs=-1)),
                [{"k": k} for k in [5, 15, 31, 61, 121]]),
    "ANN": tune("ANN", lambda hidden, alpha: scaled(MLPClassifier(hidden_layer_sizes=hidden, alpha=alpha,
                    early_stopping=True, max_iter=500, random_state=aq.SEED)),
                [{"hidden": h, "alpha": a} for h in [(16,), (64,), (64, 32)] for a in [1e-4, 1e-2]]),
}

LogReg   best {'C': 100.0}  val AUC 0.9857
LDA      best {'shrinkage': 'auto'}  val AUC 0.9798


Tree     best {'max_depth': 4}  val AUC 0.9775


RF       best {'max_depth': nan, 'min_samples_leaf': 5}  val AUC 0.9855


KNN      best {'k': 61}  val AUC 0.9723


ANN      best {'hidden': (64,), 'alpha': 0.01}  val AUC 0.9855


Class imbalance: logistic regression, the tree and the forest use `class_weight="balanced"`
(errors on the rarer class cost more). LDA, KNN and the ANN do not support class weights; for them
the imbalance is handled by the threshold chosen on validation. The validation and test sets are
never resampled.

## 6.3 Threshold on validation, then test

In [5]:
proba = {name: {k: m.predict_proba(X[k])[:, 1] for k in X} for name, m in models.items()}
thr = {name: best_threshold(y["val"], proba[name]["val"]) for name in models}
rows = []
for name in models:
    for part in ["val", "test", "test_nolock"]:
        rows.append({"model": name, "set": part, "threshold": thr[name],
                     **scores(y[part], (proba[name][part] >= thr[name]).astype(int), proba[name][part])})
res = pd.concat([pd.DataFrame(rows).set_index(["model", "set"]), base]).round(3)
res.to_csv(TAB / "06_results.csv")
test_tab = res.xs("test", level="set").sort_values("ROC_AUC", ascending=False)
test_tab

,threshold,accuracy,precision,recall,F1,F2,ROC_AUC,PR_AUC
model,,,,,,,,
RF,0.340,0.929,0.647,0.920,0.759,0.848,0.976,0.872
LogReg,0.345,0.914,0.594,0.934,0.726,0.838,0.975,0.876
ANN,0.282,0.922,0.622,0.931,0.746,0.847,0.974,0.879
LDA,0.179,0.899,0.550,0.936,0.693,0.821,0.971,0.859
Persistence,NaN,0.936,0.736,0.740,0.738,0.740,0.963,0.811
"Today's AQI, tuned threshold",NaN,0.904,0.567,0.906,0.698,0.809,0.963,0.811
Tree,0.456,0.931,0.657,0.906,0.762,0.842,0.960,0.823
KNN,0.279,0.835,0.420,0.914,0.575,0.740,0.950,0.783
Majority (always 0),NaN,0.878,0.000,0.000,0.000,0.000,0.500,0.122


In [6]:
res.xs("test_nolock", level="set").sort_values("ROC_AUC", ascending=False)

,threshold,accuracy,precision,recall,F1,F2,ROC_AUC,PR_AUC
model,,,,,,,,
LogReg,0.345,0.898,0.683,0.961,0.799,0.889,0.974,0.914
RF,0.340,0.910,0.714,0.952,0.816,0.892,0.973,0.906
ANN,0.282,0.908,0.707,0.958,0.814,0.895,0.973,0.918
LDA,0.179,0.887,0.661,0.945,0.778,0.870,0.968,0.899
Tree,0.456,0.906,0.710,0.939,0.808,0.882,0.962,0.871
"Today's AQI, tuned threshold",NaN,0.885,0.663,0.926,0.773,0.858,0.958,0.854
Persistence,NaN,0.910,0.787,0.787,0.787,0.787,0.958,0.854
KNN,0.279,0.845,0.581,0.935,0.717,0.834,0.949,0.838
Majority (always 0),NaN,0.790,0.000,0.000,0.000,0.000,0.500,0.210


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for name in models:
    fpr, tpr, _ = roc_curve(y["test"], proba[name]["test"])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y['test'], proba[name]['test']):.3f})")
    pr, rc, _ = precision_recall_curve(y["test"], proba[name]["test"])
    axes[1].plot(rc, pr, label=f"{name} (AP {average_precision_score(y['test'], proba[name]['test']):.3f})")
fpr, tpr, _ = roc_curve(y["test"], persist_score["test"])
axes[0].plot(fpr, tpr, "k--", label=f"Persistence: today's AQI (AUC {roc_auc_score(y['test'], persist_score['test']):.3f})")
pr, rc, _ = precision_recall_curve(y["test"], persist_score["test"])
axes[1].plot(rc, pr, "k--", label="Persistence: today's AQI")
axes[0].plot([0, 1], [0, 1], color="grey", lw=0.8)
axes[1].axhline(y["test"].mean(), color="grey", lw=0.8, label="positive rate (random)")
axes[0].set(xlabel="false positive rate", ylabel="true positive rate (recall)", title="ROC, test 2020")
axes[1].set(xlabel="recall", ylabel="precision", title="Precision-recall, test 2020")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
save("roc_pr")

In [8]:
names = list(models) + ["Persistence", "Today's AQI, tuned threshold"]
fig, axes = plt.subplots(2, 4, figsize=(13, 6)); axes = axes.ravel()
for ax, name in zip(axes, names):
    pred = (persist_pred["test"] if name == "Persistence" else today_pred["test"] if name.startswith("Today")
            else (proba[name]["test"] >= thr[name]).astype(int))
    sns.heatmap(confusion_matrix(y["test"], pred), annot=True, fmt="d", cbar=False, cmap="Blues", ax=ax,
                xticklabels=["pred 0", "pred 1"], yticklabels=["true 0", "true 1"])
    ax.set_title(name, fontsize=10)
save("confusion_test")

**For the report:** which model has the highest test ROC-AUC, and how much does it beat the
*tuned* today's-AQI baseline on ROC-AUC, PR-AUC and F2? That margin, not the margin over plain
persistence, is what the extra features add. Note how precision on test is lower than on
validation even for the same model: the positive rate fell from 24.5% (2019) to 12.2% (2020), and
precision depends on the positive rate while ROC-AUC does not.

## 6.4 Optional: oversampling instead of class weights (logistic regression)

In [9]:
rng = np.random.default_rng(aq.SEED)
pos_idx = np.where(y["train"] == 1)[0]; neg_idx = np.where(y["train"] == 0)[0]
over = np.concatenate([neg_idx, rng.choice(pos_idx, size=len(neg_idx), replace=True)])
C_best = models["LogReg"][-1].C
lr_os = scaled(LogisticRegression(C=C_best, max_iter=3000)).fit(X["train"].iloc[over], y["train"][over])
s_val, s_test = lr_os.predict_proba(X["val"])[:, 1], lr_os.predict_proba(X["test"])[:, 1]
t_os = best_threshold(y["val"], s_val)
print("LogReg, oversampled train:", {k: round(v, 3) for k, v in scores(y["test"], (s_test >= t_os).astype(int), s_test).items()})
print("LogReg, class weights:    ", res.loc[("LogReg", "test")].drop("threshold").to_dict())

LogReg, oversampled train: {'accuracy': 0.912, 'precision': 0.586, 'recall': 0.95, 'F1': 0.725, 'F2': 0.845, 'ROC_AUC': 0.975, 'PR_AUC': 0.875}
LogReg, class weights:     {'accuracy': 0.914, 'precision': 0.594, 'recall': 0.934, 'F1': 0.726, 'F2': 0.838, 'ROC_AUC': 0.975, 'PR_AUC': 0.876}


## 6.5 Interpretation

### Logistic regression as odds ratios

In [10]:
lr = models["LogReg"][-1]
odds = pd.DataFrame({"coef_per_SD": lr.coef_[0], "odds_ratio_per_SD": np.exp(lr.coef_[0])}, index=F)
non_city = odds[~odds.index.str.startswith("city_")].sort_values("coef_per_SD", key=abs, ascending=False)
non_city.round(3).to_csv(TAB / "06_logreg_odds_ratios.csv")
print("chosen C (inverse regularisation strength):", lr.C)
non_city.round(3)

chosen C (inverse regularisation strength): 100


,coef_per_SD,odds_ratio_per_SD
aqi_z_filled,1.900,6.686
Z_PM2.5,1.812,6.123
lag1_Z_PM2.5,-1.035,0.355
Z_PM10,1.021,2.775
lag1_aqi_z_filled,0.934,2.545
Z_CO,0.675,1.965
lag1_Z_PM10,-0.545,0.580
roll7_Z_PM2.5,0.380,1.463
lag1_Z_NO2,-0.298,0.743
month_cos,0.210,1.234


An odds ratio of 2 means: one standard deviation higher on this feature (others fixed) doubles the
odds that tomorrow is unhealthy. Correlated features (today's AQI, today's PM2.5, the 7-day mean)
share the effect, so individual ratios can even flip sign; read them as a group. Smaller C means
stronger L2 regularisation, which shrinks correlated coefficients towards each other.

### Decision tree rules (depth 3, readable version)

In [11]:
ls = info["log_standardiser"]
def to_units(feat, z):
    if feat.startswith("aqi_z") or feat.startswith("lag1_aqi"):
        return f"AQI {np.expm1(z * info['aqi_log_std'] + info['aqi_log_mean']):.0f}"
    for p in aq.POLLUTANTS:
        if feat.endswith(f"Z_{p}"):
            v = z * ls.std_[p] + ls.mean_[p]
            return f"{np.expm1(v) if p in ls.log_cols else v:.1f} (raw units)"
    return f"{z:.2f}"

small = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, class_weight="balanced",
                               random_state=aq.SEED).fit(X["train"], y["train"])
print("depth-3 tree, val ROC-AUC:", round(roc_auc_score(y["val"], small.predict_proba(X["val"])[:, 1]), 3))
t = small.tree_
def rules(node=0, depth=0):
    if t.children_left[node] == -1:
        v = t.value[node][0]; share = v[1] / v.sum()
        print("   " * depth + f"-> predict {'UNHEALTHY' if share > 0.5 else 'ok'} (weighted share unhealthy {share:.2f})")
        return
    f, z = F[t.feature[node]], t.threshold[node]
    print("   " * depth + f"if {f} <= {z:.2f}  [= {to_units(f, z)}]")
    rules(t.children_left[node], depth + 1)
    print("   " * depth + f"else ({f} > {z:.2f})")
    rules(t.children_right[node], depth + 1)
rules()
fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(small, feature_names=F, class_names=["ok", "unhealthy"], filled=True, rounded=True, fontsize=8, ax=ax)
save("tree_depth3")

depth-3 tree, val ROC-AUC: 0.974
if aqi_z_filled <= 0.31  [= AQI 185]
   if Z_PM2.5 <= 0.45  [= 84.3 (raw units)]
      if Z_PM2.5 <= 0.16  [= 67.4 (raw units)]
         -> predict ok (weighted share unhealthy 0.04)
      else (Z_PM2.5 > 0.16)
         -> predict ok (weighted share unhealthy 0.22)
   else (Z_PM2.5 > 0.45)
      if roll7_Z_PM2.5 <= 0.52  [= 89.4 (raw units)]
         -> predict UNHEALTHY (weighted share unhealthy 0.57)
      else (roll7_Z_PM2.5 > 0.52)
         -> predict UNHEALTHY (weighted share unhealthy 0.92)
else (aqi_z_filled > 0.31)
   if aqi_z_filled <= 0.63  [= AQI 229]
      if Z_PM2.5 <= 0.57  [= 92.5 (raw units)]
         -> predict UNHEALTHY (weighted share unhealthy 0.58)
      else (Z_PM2.5 > 0.57)
         -> predict UNHEALTHY (weighted share unhealthy 0.88)
   else (aqi_z_filled > 0.63)
      if aqi_z_filled <= 0.95  [= AQI 285]
         -> predict UNHEALTHY (weighted share unhealthy 0.87)
      else (aqi_z_filled > 0.95)
         -> predict UNHEALTHY (

### Random forest: permutation importance on validation

In [12]:
rf = models["RF"]
perm = permutation_importance(rf, X["val"], y["val"], scoring="roc_auc", n_repeats=5,
                              random_state=aq.SEED, n_jobs=-1)
pi = pd.Series(perm.importances_mean, index=F).sort_values(ascending=False)
pi.round(4).to_csv(TAB / "06_rf_permutation_importance.csv")
fig, ax = plt.subplots(figsize=(7, 6))
pi.head(15)[::-1].plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("drop in validation ROC-AUC when shuffled"); ax.set_title("Random forest: top 15 features")
save("rf_importance")

### Performance by city (best model by validation AUC)

In [13]:
val_auc = {nm: roc_auc_score(y["val"], proba[nm]["val"]) for nm in models}
best = max(val_auc, key=val_auc.get)
print("best model on validation:", best)
rows = []
for part in ["val", "test"]:
    df_ = S[part].assign(score=proba[best][part], pred=(proba[best][part] >= thr[best]).astype(int),
                         persist=persist_pred[part])
    for city, g in df_.groupby("City"):
        yt = g["y_unhealthy_next"].astype(int)
        rows.append({"set": part, "City": city, "days": len(g), "positive_rate": yt.mean(),
                     "ROC_AUC": roc_auc_score(yt, g["score"]) if yt.nunique() == 2 else np.nan,
                     "recall": recall_score(yt, g["pred"], zero_division=0) if yt.sum() else np.nan,
                     "recall_persistence": recall_score(yt, g["persist"], zero_division=0) if yt.sum() else np.nan})
city_perf = pd.DataFrame(rows).round(3)
city_perf.to_csv(TAB / "06_by_city.csv", index=False)
city_perf[city_perf["set"] == "val"].sort_values("positive_rate", ascending=False)

best model on validation: LogReg


,set,City,days,positive_rate,ROC_AUC,recall,recall_persistence
0,val,Ahmedabad,352,0.952,0.926,1.000,0.961
6,val,Delhi,365,0.532,0.954,0.995,0.845
12,val,Lucknow,365,0.458,0.959,0.988,0.820
13,val,Patna,359,0.443,0.976,0.943,0.881
7,val,Gurugram,364,0.423,0.963,0.981,0.838
14,val,Talcher,303,0.281,0.986,0.976,0.812
11,val,Kolkata,365,0.271,0.986,0.990,0.848
10,val,Jorapokhar,303,0.248,0.956,0.933,0.747
4,val,Brajrajnagar,322,0.208,0.963,0.955,0.701
16,val,Visakhapatnam,341,0.109,0.983,0.892,0.811


Validation (2019) is shown because 2020 has few unhealthy days in many cities (AUC undefined when a
city has no positives). Cities that are rarely unhealthy have few positives, so their recall is noisy.

### Where do models fail? Transitions vs persistent days

A *transition* is a day that is healthy today and unhealthy tomorrow: the sudden-spike case that
persistence always misses. Validation and test are pooled to have enough transitions.

In [14]:
rows = []
for part in ["val", "test"]:
    df_ = S[part]
    today = df_["unhealthy_today"]; tom = df_["y_unhealthy_next"]
    kinds = {"persistent (1 -> 1)": (today == 1) & (tom == 1), "transition (0 -> 1)": (today == 0) & (tom == 1),
             "end of episode (1 -> 0)": (today == 1) & (tom == 0)}
    for kind, mask in kinds.items():
        mask = mask.values
        for name in dict.fromkeys([best, "LogReg", "RF", "Persistence", "Today's AQI, tuned threshold"]):
            pred = (persist_pred[part] if name == "Persistence" else today_pred[part] if name.startswith("Today")
                    else (proba[name][part] >= thr[name]).astype(int))
            rows.append({"set": part, "case": kind, "model": name, "days": int(mask.sum()),
                         "share_predicted_unhealthy": pred[mask].mean() if mask.sum() else np.nan})
err = pd.DataFrame(rows).groupby(["case", "model"]).apply(
    lambda g: pd.Series({"days": g["days"].sum(),
                         "share_predicted_unhealthy": np.average(g["share_predicted_unhealthy"], weights=g["days"])}),
    include_groups=False).round(3)
err.to_csv(TAB / "06_transitions.csv")
err

days  share_predicted_unhealthy
case                    model                                                          
end of episode (1 -> 0) LogReg                         309.0                      0.909
                        Persistence                    309.0                      1.000
                        RF                             309.0                      0.964
                        Today's AQI, tuned threshold   309.0                      1.000
persistent (1 -> 1)     LogReg                        1498.0                      0.997
                        Persistence                   1498.0                      1.000
                        RF                            1498.0                      0.999
                        Today's AQI, tuned threshold  1498.0                      1.000
transition (0 -> 1)     LogReg                         297.0                      0.805
                        Persistence                    297.0                      0.000
                        RF                             297.0                      0.747
                        Today's AQI, tuned threshold   297.0                      0.707

**For the report:** for transitions, the share predicted unhealthy is the recall on sudden spikes.
Compare the models with "today's AQI, tuned threshold", not only with plain persistence: a
lower alarm threshold alone catches many transitions (days just under 200 that tip over).
For 'end of episode', it is the false-alarm rate on the day after an episode ends. Models gain
over persistence mainly by catching some transitions, at the cost of extra false alarms after
episodes. If transition recall is low, say so: without weather forecasts, spikes are hard to see
coming.

## 6.6 Sensitivity: without Ahmedabad

In [15]:
d_na, info_na = aq.build_dataset(exclude_cities=["Ahmedabad"])
F_na = info_na["rq2_features"]
c_na = d_na.dropna(subset=["y_unhealthy_next"])
T = {p: c_na[c_na["period"] == p] for p in ["train", "val", "test"]}
rows = []
for name in ["LogReg", "RF", best]:
    m = models[name]
    from sklearn.base import clone
    m2 = clone(m).fit(T["train"][F_na], T["train"]["y_unhealthy_next"].astype(int))
    for part in ["val", "test"]:
        rows.append({"model": name, "set": part, "ROC_AUC_no_Ahmedabad":
                     roc_auc_score(T[part]["y_unhealthy_next"], m2.predict_proba(T[part][F_na])[:, 1]),
                     "ROC_AUC_all": roc_auc_score(y[part], proba[name][part])})
sens = pd.DataFrame(rows).drop_duplicates(["model", "set"]).round(3)
sens.to_csv(TAB / "06_sensitivity_no_ahmedabad.csv", index=False)
sens

,model,set,ROC_AUC_no_Ahmedabad,ROC_AUC_all
0,LogReg,val,0.983,0.986
1,LogReg,test,0.975,0.975
2,RF,val,0.983,0.986
3,RF,test,0.976,0.976
